# Train model gương chiếu hậu + gán nháp vòng sau

**Chuẩn bị (1 lần mỗi vòng):** tạo 1 Kaggle Dataset (Datasets → New Dataset) gồm 3 file:

| File | Lấy ở đâu |
|---|---|
| `1_anh_xe_may.zip` | file ảnh đã dùng để tạo task CVAT |
| `VN_street_motorbike_300_image_labels.json` | file thông tin bộ ảnh |
| `cvat_export.zip` | CVAT → Actions → Export task dataset → **CVAT for images 1.1** (đổi tên thành `cvat_export.zip`) |

Kaggle tự giải nén file zip — không sao, notebook đọc được cả 2 dạng.

**Chạy:** Add Input → chọn dataset trên → Settings: **Accelerator = GPU T4 x2**, **Internet = On** → **Run All** (~20–40 phút).

**Kết quả** (tab Output, bên phải):
- `mirror_best.pt` — model gương
- `prelabel_cvat11.zip` — nhãn nháp: CVAT → Actions → **Upload annotations** → **CVAT 1.1** → duyệt & sửa


In [ ]:
!pip install -q ultralytics==8.4.168

In [ ]:
%%writefile prepare_mirror_dataset.py
"""
Chuẩn bị dữ liệu gương chiếu hậu (Phần B, docs/PLAN_GUONG_VA_NHIEU_NGUOI_VAO_RA.md).

Bước 1: tạo task CVAT để gán nhãn gương từ bộ VN_street_motorbike_300
    python scripts/prepare_mirror_dataset.py cvat-task \
        --zip VN_street_motorbike_300_CVAT_1.1.zip \
        --labels VN_street_motorbike_300_image_labels.json \
        --out datasets/vn_mirror/vn_mirror_cvat_task.zip

    -> Import zip vào CVAT (Create task -> upload ảnh; Actions -> Upload
       annotations -> "CVAT 1.1"). Trong CVAT:
       * Vẽ box `mirror` quanh MẶT GƯƠNG (không vẽ cả cần gương).
       * Gắn đúng 1 tag trạng thái cho mỗi ảnh: mirror_both, mirror_only_left,
         mirror_only_right, mirror_none, mirror_occluded. Trái/phải theo NGƯỜI
         LÁI: ảnh chụp phía trước xe thì gương trái người lái nằm bên PHẢI ảnh.
       * Ảnh xe quá nhỏ đã được gắn sẵn tag `too_small` — bỏ qua, không vẽ.

Bước 2: sau khi gán nhãn xong, export "CVAT for images 1.1" rồi chuyển sang YOLO
    python scripts/prepare_mirror_dataset.py to-yolo \
        --cvat exported_cvat.zip \
        --labels VN_street_motorbike_300_image_labels.json \
        --out datasets/vn_mirror

    -> datasets/vn_mirror/{images,labels}/{train,val} + data.yaml (1 lớp: mirror).
       Chia train/val theo ẢNH GỐC (source_original_image): 2 crop cắt từ cùng
       1 khung hình luôn nằm cùng 1 phía, tránh "học thuộc" làm điểm val đẹp giả.

Nguồn: Kaggle "Transportation in Ho Chi Minh City" (khoakait), giấy phép CC BY 4.0
— phải ghi nguồn khi dùng/công bố model.
"""
from __future__ import annotations

import argparse
import hashlib
import io
import json
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

MIN_BIKE_WIDTH_PX = 60
STATE_TAGS = ['mirror_both', 'mirror_only_left', 'mirror_only_right', 'mirror_none', 'mirror_occluded']
SKIP_TAGS = {'too_small'}
VAL_PERCENT = 15


class Source:
    """Một zip HOẶC một thư mục (Kaggle tự giải nén zip khi upload dataset),
    tra file theo TÊN (bỏ đường dẫn con)."""

    def __init__(self, path: str):
        self.path = Path(path)
        if self.path.is_dir():
            self._zip = None
            files = [p for p in self.path.rglob('*') if p.is_file()]
            self._names = {p.name: p for p in files}
        else:
            self._zip = zipfile.ZipFile(self.path)
            self._names = {Path(n).name: n for n in self._zip.namelist() if not n.endswith('/')}

    def names(self) -> list[str]:
        return list(self._names)

    def __contains__(self, name: str) -> bool:
        return name in self._names

    def read(self, name: str) -> bytes:
        target = self._names[name]
        return self._zip.read(target) if self._zip is not None else target.read_bytes()


def _load_meta(labels_path: str) -> dict:
    return {row['file_name']: row for row in json.loads(Path(labels_path).read_text(encoding='utf-8'))}


def _labels_xml(parent: ET.Element, labels: list[tuple[str, str]]) -> None:
    node = ET.SubElement(parent, 'labels')
    for name, kind in labels:
        label = ET.SubElement(node, 'label')
        ET.SubElement(label, 'name').text = name
        ET.SubElement(label, 'type').text = kind
        ET.SubElement(label, 'attributes')


def build_cvat_task(src_zip: str, labels_path: str, out_zip: str) -> dict:
    meta = _load_meta(labels_path)
    with zipfile.ZipFile(src_zip) as zin:
        source = ET.fromstring(zin.read('annotations.xml'))
        root = ET.Element('annotations')
        ET.SubElement(root, 'version').text = '1.1'
        task = ET.SubElement(ET.SubElement(root, 'meta'), 'task')
        ET.SubElement(task, 'name').text = 'VN_mirror_labeling'
        _labels_xml(task, [('motorbike', 'rectangle'), ('mirror', 'rectangle')]
                    + [(t, 'tag') for t in STATE_TAGS + sorted(SKIP_TAGS)]
                    + [('front_or_front_oblique', 'tag'), ('side_or_side_oblique', 'tag'),
                       ('rear_or_rear_oblique', 'tag')])
        stats = {'images': 0, 'too_small': 0}
        for image in source.iter('image'):
            out = ET.SubElement(root, 'image', dict(image.attrib))
            for child in image:
                out.append(child)
            name = Path(image.get('name')).name
            x1, _, x2, _ = meta.get(name, {}).get('motorbike_bbox_xyxy', (0, 0, 1e9, 0))
            if x2 - x1 < MIN_BIKE_WIDTH_PX:
                ET.SubElement(out, 'tag', {'label': 'too_small', 'source': 'auto'})
                stats['too_small'] += 1
            stats['images'] += 1
        Path(out_zip).parent.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as zout:
            for item in zin.namelist():
                if item.startswith('images/') and not item.endswith('/'):
                    zout.writestr(item, zin.read(item))
            buf = io.BytesIO()
            ET.ElementTree(root).write(buf, encoding='utf-8', xml_declaration=True)
            zout.writestr('annotations.xml', buf.getvalue())
    return stats


def _split(group_key: str) -> str:
    digest = int(hashlib.sha1(group_key.encode('utf-8')).hexdigest(), 16)
    return 'val' if digest % 100 < VAL_PERCENT else 'train'


def split_of(name: str, meta: dict) -> str:
    """'train' / 'val'. Theo ẢNH GỐC: 2 crop cùng khung hình luôn cùng phía.
    Tập 'val' cũng là ĐỀ THI: phải gán tay 100%, không bao giờ gán nháp."""
    return _split(meta.get(name, {}).get('source_original_image', name))


def held_out(labels_path: str) -> list[str]:
    """Ảnh thuộc đề thi (val), trừ ảnh xe quá nhỏ — cần gán tay trước tiên."""
    meta = _load_meta(labels_path)
    out = []
    for name, row in sorted(meta.items()):
        x1, _, x2, _ = row.get('motorbike_bbox_xyxy', (0, 0, 1e9, 0))
        if split_of(name, meta) == 'val' and x2 - x1 >= MIN_BIKE_WIDTH_PX:
            out.append(name)
    return out


def cvat_to_yolo(cvat_zip: str, labels_path: str, out_dir: str, images: str | None = None) -> dict:
    """images: zip/thư mục ảnh khi export CVAT KHÔNG kèm ảnh (mặc định lấy
    ảnh trong chính file export)."""
    meta = _load_meta(labels_path)
    out = Path(out_dir)
    stats = {'train': 0, 'val': 0, 'skipped': 0, 'mirror_boxes': 0, 'unlabeled': 0}
    export = Source(cvat_zip)
    pictures = Source(images) if images else export
    root = ET.fromstring(export.read('annotations.xml'))
    for image in root.iter('image'):
        name = Path(image.get('name')).name
        tags = {t.get('label') for t in image.findall('tag')}
        if tags & SKIP_TAGS:
            stats['skipped'] += 1
            continue
        if not tags & set(STATE_TAGS):
            # Chưa gán trạng thái = chưa duyệt -> không đưa vào train
            # (ảnh không box sẽ bị hiểu nhầm là "chắc chắn không có gương").
            stats['unlabeled'] += 1
            continue
        w, h = float(image.get('width')), float(image.get('height'))
        lines = []
        for box in image.findall('box'):
            if box.get('label') != 'mirror':
                continue
            x1, y1, x2, y2 = (float(box.get(k)) for k in ('xtl', 'ytl', 'xbr', 'ybr'))
            lines.append(f'0 {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} '
                         f'{(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}')
        if name not in pictures:
            raise FileNotFoundError(f'Không thấy ảnh {name} — truyền --images 1_anh_xe_may.zip')
        split = split_of(name, meta)
        (out / 'images' / split).mkdir(parents=True, exist_ok=True)
        (out / 'labels' / split).mkdir(parents=True, exist_ok=True)
        (out / 'images' / split / name).write_bytes(pictures.read(name))
        (out / 'labels' / split / (Path(name).stem + '.txt')).write_text('\n'.join(lines))
        stats[split] += 1
        stats['mirror_boxes'] += len(lines)
    (out / 'data.yaml').write_text(
        f'path: {out.resolve().as_posix()}\ntrain: images/train\nval: images/val\nnames:\n  0: mirror\n',
        encoding='utf-8')
    return stats


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = parser.add_subparsers(dest='cmd', required=True)
    task = sub.add_parser('cvat-task')
    task.add_argument('--zip', required=True)
    task.add_argument('--labels', required=True)
    task.add_argument('--out', default='datasets/vn_mirror/vn_mirror_cvat_task.zip')
    yolo = sub.add_parser('to-yolo')
    yolo.add_argument('--cvat', required=True)
    yolo.add_argument('--labels', required=True)
    yolo.add_argument('--out', default='datasets/vn_mirror')
    yolo.add_argument('--images', help='zip/thư mục ảnh nếu export CVAT không kèm ảnh')
    test = sub.add_parser('test-set', help='in danh sách ảnh đề thi (gán tay trước)')
    test.add_argument('--labels', required=True)
    args = parser.parse_args()
    if args.cmd == 'cvat-task':
        print(build_cvat_task(args.zip, args.labels, args.out))
    elif args.cmd == 'test-set':
        print('\n'.join(held_out(args.labels)))
    else:
        print(cvat_to_yolo(args.cvat, args.labels, args.out, args.images))


if __name__ == '__main__':
    main()


In [ ]:
%%writefile prelabel_mirror.py
"""
Gán NHÁP nhãn gương bằng model vừa train, để nạp ngược vào CVAT rồi duyệt/sửa.

    python scripts/prelabel_mirror.py --model best.pt \
        --cvat export_cvat11.zip --images 1_anh_xe_may.zip \
        --labels VN_street_motorbike_300_image_labels.json \
        --out prelabel_cvat11.zip

--cvat là file export MỚI NHẤT từ CVAT ("CVAT for images 1.1"). CVAT "Upload
annotations" GHI ĐÈ toàn bộ nhãn của task, nên file đầu ra giữ nguyên MỌI nhãn
đã có và chỉ THÊM box gương nháp vào ảnh:
  - chưa gắn tag trạng thái (chưa duyệt),
  - không phải ảnh `too_small`,
  - KHÔNG thuộc đề thi (tập val) — đề thi luôn gán tay 100%, nếu không điểm
    model sẽ đẹp giả vì chấm trên chính nhãn do nó tự sinh.
Ngưỡng --conf để thấp (mặc định 0.15): xóa box thừa nhanh hơn vẽ box bị sót.
"""
from __future__ import annotations

import argparse
import io
import sys
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
from prepare_mirror_dataset import SKIP_TAGS, STATE_TAGS, Source, _load_meta, split_of  # noqa: E402


def images_to_prelabel(root: ET.Element, meta: dict) -> list[str]:
    out = []
    for image in root.iter('image'):
        name = Path(image.get('name')).name
        tags = {t.get('label') for t in image.findall('tag')}
        if tags & (SKIP_TAGS | set(STATE_TAGS)) or split_of(name, meta) == 'val':
            continue
        out.append(name)
    return out


def merge_predictions(root: ET.Element, predictions: dict) -> int:
    """predictions: {tên ảnh: [(x1, y1, x2, y2, conf), ...]}. Xóa box gương
    nháp cũ (source="auto") của ảnh đó rồi thêm box mới. Trả số box đã thêm."""
    added = 0
    for image in root.iter('image'):
        boxes = predictions.get(Path(image.get('name')).name)
        if boxes is None:
            continue
        for old in [b for b in image.findall('box') if b.get('label') == 'mirror' and b.get('source') == 'auto']:
            image.remove(old)
        for x1, y1, x2, y2, _conf in boxes:
            ET.SubElement(image, 'box', {
                'label': 'mirror', 'source': 'auto', 'occluded': '0', 'z_order': '0',
                'xtl': f'{x1:.2f}', 'ytl': f'{y1:.2f}', 'xbr': f'{x2:.2f}', 'ybr': f'{y2:.2f}'})
            added += 1
    return added


def predict(model_path: str, pictures: Source, names: list[str], conf: float) -> dict:
    import cv2
    import numpy as np
    from ultralytics import YOLO
    model = YOLO(model_path)
    out = {}
    for name in names:
        frame = cv2.imdecode(np.frombuffer(pictures.read(name), np.uint8), cv2.IMREAD_COLOR)
        result = model.predict(frame, conf=conf, imgsz=320, verbose=False)[0]
        out[name] = [(*map(float, b.xyxy[0].tolist()), float(b.conf[0])) for b in result.boxes]
    return out


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('--model', required=True)
    parser.add_argument('--cvat', required=True, help='export CVAT 1.1 mới nhất')
    parser.add_argument('--images', required=True, help='zip/thư mục ảnh (1_anh_xe_may.zip)')
    parser.add_argument('--labels', required=True, help='VN_street_motorbike_300_image_labels.json')
    parser.add_argument('--out', default='prelabel_cvat11.zip')
    parser.add_argument('--conf', type=float, default=0.15)
    args = parser.parse_args()

    root = ET.fromstring(Source(args.cvat).read('annotations.xml'))
    names = images_to_prelabel(root, _load_meta(args.labels))
    added = merge_predictions(root, predict(args.model, Source(args.images), names, args.conf))
    buf = io.BytesIO()
    ET.ElementTree(root).write(buf, encoding='utf-8', xml_declaration=True)
    with zipfile.ZipFile(args.out, 'w', zipfile.ZIP_DEFLATED) as z:
        z.writestr('annotations.xml', buf.getvalue())
    print({'images_prelabeled': len(names), 'mirror_boxes_added': added, 'out': args.out})


if __name__ == '__main__':
    main()


In [ ]:
import glob, os

def find(*patterns):
    for pat in patterns:
        hits = sorted(glob.glob(f'/kaggle/input/**/{pat}', recursive=True))
        if hits:
            return hits[0]
    raise FileNotFoundError(f'Không thấy {patterns} trong /kaggle/input — kiểm tra lại dataset đã Add Input chưa')

LABELS = find('*image_labels*.json')
IMAGES = find('1_anh_xe_may.zip', '1_anh_xe_may')
CVAT = find('cvat_export*.zip', 'cvat_export*')
print('labels:', LABELS); print('images:', IMAGES); print('cvat  :', CVAT)


In [ ]:
from prepare_mirror_dataset import cvat_to_yolo, held_out

stats = cvat_to_yolo(CVAT, LABELS, '/kaggle/working/vn_mirror', IMAGES)
print(stats)
if stats['val'] == 0:
    print('ĐỀ THI (gán tay các ảnh này trước):')
    print('\n'.join(held_out(LABELS)))
    raise SystemExit('Chưa có ảnh đề thi nào được gán nhãn — gán tay các ảnh ở danh sách trên rồi export lại')
if stats['train'] < 20:
    raise SystemExit(f"Mới có {stats['train']} ảnh train — gán thêm cho đủ ít nhất 20 ảnh rồi chạy lại")


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(
    data='/kaggle/working/vn_mirror/data.yaml',
    epochs=80, imgsz=320, batch=32, device=0,
    fliplr=0.5,          # 1 lớp `mirror`, trái/phải tính bằng hình học nên lật ngang vẫn đúng
    patience=20, project='/kaggle/working/runs', name='mirror', exist_ok=True,
)


In [ ]:
import shutil

best = '/kaggle/working/runs/mirror/weights/best.pt'
shutil.copy(best, '/kaggle/working/mirror_best.pt')
metrics = YOLO(best).val(data='/kaggle/working/vn_mirror/data.yaml', imgsz=320, verbose=False)
p, r, m50 = metrics.box.mp, metrics.box.mr, metrics.box.map50
print(f'Trên ĐỀ THI: precision={p:.2f}  recall={r:.2f}  mAP50={m50:.2f}')
print('precision = trong các box model vẽ, bao nhiêu là gương thật')
print('recall    = trong các gương thật, model tìm được bao nhiêu')
print('Mục tiêu trước khi dùng thật: mAP50 >= 0.80 trên ảnh quay tại cổng.')


In [ ]:
import subprocess, sys

subprocess.run([sys.executable, 'prelabel_mirror.py', '--model', '/kaggle/working/mirror_best.pt',
                '--cvat', CVAT, '--images', IMAGES, '--labels', LABELS,
                '--out', '/kaggle/working/prelabel_cvat11.zip'], check=True)
print('Tải prelabel_cvat11.zip ở tab Output → CVAT → Actions → Upload annotations → CVAT 1.1')
